In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision


In [4]:
from torchvision.datasets import CIFAR10 
import torchvision.transforms as transforms

transform  = transforms.Compose ([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train = CIFAR10(root="./data",train=True,download=True,transform=transform)
test = CIFAR10(root="./data",train=False,download=True,transform=transform)

c:\Users\jinju\anaconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
print(train.root)

./data


100%|██████████| 170M/170M [21:51<00:00, 130kB/s]    
c:\Users\jinju\anaconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")

In [6]:
from torch.utils.data import DataLoader,Dataset

trainloader = DataLoader(train,batch_size=64,shuffle=True)
testloader = DataLoader(test,batch_size=64)



In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.convo_layer = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1), # channel, out channel
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernal , stride

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fully_conn_layer = nn.Sequential(
             
            nn.Flatten(),
            nn.Linear(4*4*128,256), #Output=SN−F+2P​+1= 256
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward (self,x):
            x = self.convo_layer(x)
            x = x.view(x.size(0), -1) # flattening
            output = self.fully_conn_layer(x)

            return output

In [13]:
model = CNN()
critarion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [17]:

epochs = 10

best_epoch =float("inf") 

for epoch in range(epochs):
    epoch_loss = 0.0
    correct = 0.0
    total = 0.0
    
    for Image,labels in trainloader:
        optimizer.zero_grad()

        output = model(Image)
        loss = critarion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_loss +=loss.item()
    print("epoch : ",epoch,"||","epoch_train_lost :",epoch_loss/len(trainloader))

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
         for Image,labels in testloader:
             output_test = model.forward(Image)
             loss = critarion(output_test,labels)
             test_loss += loss.item()
             
             _,predict = torch.max(output_test,1)
             correct += (predict == labels).sum().item()
             total += labels.size(0) 


    print("test_loss :",test_loss/len(testloader),"test_accuracy : ", correct/total)

    if test_loss < best_epoch :
        best_epoch = test_loss
        torch.save(model.state_dict(),"best_cnn.pt")




epoch :  0 || epoch_train_lost : 0.10152558704166462
test_loss : 1.2679738687102202 test_accuracy :  0.7419
epoch :  1 || epoch_train_lost : 0.10069269639656633
test_loss : 1.4116274182963524 test_accuracy :  0.7392
epoch :  2 || epoch_train_lost : 0.09352757914713525
test_loss : 1.4647653055418828 test_accuracy :  0.7448
epoch :  3 || epoch_train_lost : 0.07886225248203443
test_loss : 1.4791015337227256 test_accuracy :  0.7385
epoch :  4 || epoch_train_lost : 0.08018040001604354
test_loss : 1.6826848370634067 test_accuracy :  0.7396
epoch :  5 || epoch_train_lost : 0.0750884766987575
test_loss : 1.646162415765653 test_accuracy :  0.7453
epoch :  6 || epoch_train_lost : 0.0722942004487028
test_loss : 1.675710679998823 test_accuracy :  0.7437
epoch :  7 || epoch_train_lost : 0.07019441447023045
test_loss : 1.682122154220654 test_accuracy :  0.7491
epoch :  8 || epoch_train_lost : 0.06570854956687451
test_loss : 1.806534693878927 test_accuracy :  0.7467
epoch :  9 || epoch_train_lost : 0